# Search and Planning — Exercise Solutions

Worked answers for the separate exercises. Alternative valid traces may choose a different route when multiple routes tie.

In [1]:
import importlib.util
import sys
from pathlib import Path
import matplotlib

implementation_path = Path("07_Artificial_Intelligence_Fundamentals/03_Search_and_Planning_Visual_Examples/05_search_algorithms.py")
spec = importlib.util.spec_from_file_location("search_algorithms", implementation_path)
if spec is None or spec.loader is None:
    raise RuntimeError(f"Cannot load search implementation: {implementation_path}")
search = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = search
spec.loader.exec_module(search)
print(f"Python {sys.version.split()[0]} | Matplotlib {matplotlib.__version__}")

Python 3.13.14 | Matplotlib 3.10.8


## Solution 1 — BFS trace

Starting queue `[A]`. Expand A, queue becomes `[B, C]`; expand B, queue `[C, D]`; expand C, D is already discovered; expand D, queue `[goal]`; expand goal and stop. One route is A-B-D-goal. If C is explored before B or adjacency order differs, the chosen equal-length route can differ.

## Solution 2 — Manual cost

A-B costs 2; B-D costs 2; D-G costs 3. Total \(2+2+3=7\) synthetic cost units.

In [2]:
path = ["A", "B", "D", "G"]
edge_cost = {(u, v): w for u, edges in search.SAMPLE_GRAPH.items() for v, w in edges}
print("Manual path total:", sum(edge_cost[(u, v)] for u, v in zip(path, path[1:])))

Manual path total: 7


## Solution 3 — BFS versus Dijkstra

BFS chooses a fewest-edge path and ignores weights. Dijkstra chooses the least sum. In this graph BFS uses the direct A-G edge (one edge, cost 9), while Dijkstra uses A-B-D-G (three edges, cost 7). Changing costs can alter Dijkstra's preferred route but not BFS unless connectivity changes.

In [3]:
print("BFS:", search.breadth_first_search(search.SAMPLE_GRAPH, "A", "G"))
print("Dijkstra:", search.dijkstra(search.SAMPLE_GRAPH, "A", "G"))

BFS: ['A', 'G']
Dijkstra: (['A', 'B', 'D', 'G'], 7.0)


## Solution 4 — Heuristic

Compare each estimate to the exact shortest remaining cost computed from that node. No estimate should exceed that cost for admissibility. Admissibility does not by itself eliminate graph-search implementation concerns; consistency or reopening handling matters.

In [4]:
checks = []
for node in search.SAMPLE_GRAPH:
    _, exact = search.dijkstra(search.SAMPLE_GRAPH, node, "G")
    estimate = search.HEURISTIC[node]
    checks.append(estimate <= exact)
    print(f"{node}: h={estimate}, exact={exact:g}, admissible={estimate <= exact}")
print("All admissible:", all(checks))

A: h=5, exact=7, admissible=True
B: h=4, exact=5, admissible=True
C: h=6, exact=8, admissible=True
D: h=2, exact=3, admissible=True
E: h=1, exact=1, admissible=True
F: h=2, exact=2, admissible=True
G: h=0, exact=0, admissible=True
All admissible: True


## Solution 5 — Invalid and unreachable cases

Negative edges are rejected because Dijkstra's assumption is violated. An unknown start or goal is rejected as malformed task input. A disconnected goal returns `(None, infinity)`, representing no path. A negative or non-finite heuristic is rejected because this implementation expects finite nonnegative estimates.

In [5]:
cases = [
    lambda: search.dijkstra({"A": [("B", -1)], "B": []}, "A", "B"),
    lambda: search.dijkstra({"A": []}, "missing", "A"),
    lambda: search.dijkstra({"A": [], "G": []}, "A", "G"),
    lambda: search.a_star({"A": [("G", 1)], "G": []}, "A", "G", lambda _: -1),
]
for run in cases:
    try:
        print(run())
    except ValueError as error:
        print(type(error).__name__ + ":", error)

ValueError: edge costs must be finite and non-negative
ValueError: start and goal must be graph nodes
(None, inf)
ValueError: heuristic values must be finite and non-negative


## Solution 6 — Network planning design

One safe design uses states describing which synthetic evidence has been collected and whether a hypothesis is supported. Edges represent read-only inspections with abstract time costs. The goal is enough evidence for an operator review, not an automatic repair. Device writes are prohibited; any later action must use independently authenticated change approval. Evaluate evidence usefulness, missed cases, review burden, and abstention on time-separated scenarios.

## Knowledge check answers

1. BFS is cost-optimal when each edge has equal cost. 2. Dijkstra requires nonnegative edge costs. 3. \(h(n)\) estimates cheapest remaining cost. 4. Overestimation can make A* prefer a suboptimal route. 5. It guarantees a result only relative to the encoded graph, cost, and algorithm assumptions.